# CNN vs Transformer for FMA Small Genre Classification

This notebook compares a CNN baseline against the existing transformer on the same cleaned `fma_small` split.

In [ ]:
%pip install torch torchaudio pandas numpy tqdm soundfile

In [41]:
import os
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import DataLoader, Dataset

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_CANDIDATES:
    audio_dir = candidate / "fma_small"
    metadata_path = candidate / "fma_metadata" / "tracks.csv"
    if audio_dir.exists() and metadata_path.exists():
        PROJECT_DIR = candidate.resolve()
        AUDIO_DIR = audio_dir.resolve()
        METADATA_PATH = metadata_path.resolve()
        break
else:
    raise FileNotFoundError("Could not find fma_small/ and fma_metadata/tracks.csv.")

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

GPU_TOTAL_GB = 0.0
if device == "cuda":
    GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

# A 1.6-2 GB GPU needs a much more conservative pipeline.
LOW_VRAM_MODE = device == "cuda" and GPU_TOTAL_GB < 3.0
PREPROCESS_DEVICE = "cpu" if LOW_VRAM_MODE else device

print({
    "device": device,
    "gpu_total_gb": round(GPU_TOTAL_GB, 2),
    "low_vram_mode": LOW_VRAM_MODE,
    "preprocess_device": PREPROCESS_DEVICE,
})

{'device': 'cuda', 'gpu_total_gb': 1.64, 'low_vram_mode': True, 'preprocess_device': 'cpu'}


## Load and Clean Metadata

In [42]:
AUDIO_VALIDATION_CACHE = PROJECT_DIR / "fma_small_audio_validation.csv"


def track_id_to_audio_path(track_id: int) -> Path:
    track_id = int(track_id)
    filename = f"{track_id:06d}.mp3"
    return AUDIO_DIR / filename[:3] / filename


def can_decode_audio(path: Path) -> tuple[bool, str]:
    try:
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        if sample_rate <= 0 or audio.size == 0:
            return False, "empty_audio"
        return True, ""
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


def filter_decodable_audio(frame: pd.DataFrame) -> pd.DataFrame:
    paths = frame["audio_path"].map(str)
    validation = None

    if AUDIO_VALIDATION_CACHE.exists():
        cached = pd.read_csv(AUDIO_VALIDATION_CACHE)
        if {"audio_path", "is_valid", "error"}.issubset(cached.columns):
            cached = cached.drop_duplicates(subset=["audio_path"], keep="last")
            if set(paths).issubset(set(cached["audio_path"])):
                validation = cached.set_index("audio_path")

    if validation is None:
        rows = []
        for path in tqdm(frame["audio_path"], desc="Validating audio files"):
            is_valid, error = can_decode_audio(path)
            rows.append({"audio_path": str(path), "is_valid": is_valid, "error": error})
        validation = pd.DataFrame(rows).drop_duplicates(subset=["audio_path"], keep="last")
        validation.to_csv(AUDIO_VALIDATION_CACHE, index=False)
        validation = validation.set_index("audio_path")

    filtered = frame.copy()
    filtered["audio_path_str"] = filtered["audio_path"].map(str)
    filtered = filtered.join(validation, on="audio_path_str")
    bad_rows = filtered[~filtered["is_valid"].fillna(False)].copy()
    if not bad_rows.empty:
        print(f"Dropping {len(bad_rows):,} corrupt/unreadable files.")
        display(bad_rows[["track_id", "split", "genre", "audio_path", "error"]].head(10))

    return filtered[filtered["is_valid"].fillna(False)].drop(columns=["audio_path_str", "is_valid", "error"]).copy()


def load_fma_small_metadata(validate_audio: bool = True) -> pd.DataFrame:
    tracks = pd.read_csv(METADATA_PATH, index_col=0, header=[0, 1])
    frame = pd.DataFrame({
        "track_id": tracks.index.astype(int),
        "subset": tracks[("set", "subset")],
        "split": tracks[("set", "split")],
        "genre": tracks[("track", "genre_top")],
        "duration": tracks[("track", "duration")],
    })
    frame["audio_path"] = frame["track_id"].apply(track_id_to_audio_path)
    frame = frame[
        (frame["subset"] == "small")
        & frame["genre"].notna()
        & frame["audio_path"].map(lambda path: path.exists())
    ].copy()
    if validate_audio:
        frame = filter_decodable_audio(frame)
    return frame


metadata = load_fma_small_metadata(validate_audio=True)
print(f"Usable clips: {len(metadata):,}")
metadata.head()

Dropping 6 corrupt/unreadable files.


,track_id,split,genre,audio_path,error
track_id,,,,,
98565,98565,training,Hip-Hop,/home/lilyd/acml-project/fma_small/098/098565.mp3,LibsndfileError: Unspecified internal error.
98567,98567,training,Hip-Hop,/home/lilyd/acml-project/fma_small/098/098567.mp3,LibsndfileError: Unspecified internal error.
98569,98569,training,Hip-Hop,/home/lilyd/acml-project/fma_small/098/098569.mp3,LibsndfileError: Unspecified internal error.
99134,99134,training,Electronic,/home/lilyd/acml-project/fma_small/099/099134.mp3,LibsndfileError: Error opening '/home/lilyd/ac...
108925,108925,training,Rock,/home/lilyd/acml-project/fma_small/108/108925.mp3,LibsndfileError: Error opening '/home/lilyd/ac...
133297,133297,training,Experimental,/home/lilyd/acml-project/fma_small/133/133297.mp3,LibsndfileError: Error opening '/home/lilyd/ac...


Usable clips: 7,994


,track_id,subset,split,genre,duration,audio_path
track_id,,,,,,
2,2,small,training,Hip-Hop,168,/home/lilyd/acml-project/fma_small/000/000002.mp3
5,5,small,training,Hip-Hop,206,/home/lilyd/acml-project/fma_small/000/000005.mp3
10,10,small,training,Pop,161,/home/lilyd/acml-project/fma_small/000/000010.mp3
140,140,small,training,Folk,253,/home/lilyd/acml-project/fma_small/000/000140.mp3
141,141,small,training,Folk,182,/home/lilyd/acml-project/fma_small/000/000141.mp3


## Balanced Split

In [43]:
MAX_TRAIN_PER_GENRE = 250
MAX_EVAL_PER_GENRE = 100


def cap_per_genre(frame: pd.DataFrame, max_per_genre: int | None) -> pd.DataFrame:
    if frame.empty:
        return frame.reset_index(drop=True)
    if max_per_genre is None:
        return frame.sample(frac=1, random_state=SEED).reset_index(drop=True)
    sampled_groups = [
        group.sample(min(len(group), max_per_genre), random_state=SEED)
        for _, group in frame.groupby("genre", sort=False)
    ]
    return pd.concat(sampled_groups).sample(frac=1, random_state=SEED).reset_index(drop=True)


train_df = cap_per_genre(metadata[metadata["split"] == "training"], MAX_TRAIN_PER_GENRE)
val_df = cap_per_genre(metadata[metadata["split"] == "validation"], MAX_EVAL_PER_GENRE)
test_df = cap_per_genre(metadata[metadata["split"] == "test"], MAX_EVAL_PER_GENRE)

genres = sorted(train_df["genre"].unique())
genre_to_idx = {genre: idx for idx, genre in enumerate(genres)}
idx_to_genre = {idx: genre for genre, idx in genre_to_idx.items()}

for frame in (train_df, val_df, test_df):
    frame["label"] = frame["genre"].map(genre_to_idx).astype(int)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(genre_to_idx)

Train: 2,000 | Val: 800 | Test: 800
{'Electronic': 0, 'Experimental': 1, 'Folk': 2, 'Hip-Hop': 3, 'Instrumental': 4, 'International': 5, 'Pop': 6, 'Rock': 7}


## Dataset and DataLoaders

In [44]:
SAMPLE_RATE = 16_000
CLIP_SECONDS = 10
NUM_SAMPLES = SAMPLE_RATE * CLIP_SECONDS
N_FFT = 400
HOP_LENGTH = 160
N_MELS = 64


class FMASmallWaveformDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, train: bool = False):
        self.frame = frame.reset_index(drop=True)
        self.train = train

    def __len__(self) -> int:
        return len(self.frame)

    def _load_audio(self, path: Path) -> torch.Tensor:
        try:
            audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        except Exception as exc:
            raise RuntimeError(f"Failed to decode audio file: {path}") from exc

        if sample_rate <= 0 or audio.size == 0:
            raise RuntimeError(f"Decoded empty audio file: {path}")
        if not np.isfinite(audio).all():
            raise RuntimeError(f"Decoded non-finite audio values: {path}")

        waveform = torch.from_numpy(audio.T)
        waveform = waveform.mean(dim=0, keepdim=True)

        if sample_rate != SAMPLE_RATE:
            waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)

        if waveform.shape[1] > NUM_SAMPLES:
            if self.train:
                start = random.randint(0, waveform.shape[1] - NUM_SAMPLES)
            else:
                start = (waveform.shape[1] - NUM_SAMPLES) // 2
            waveform = waveform[:, start:start + NUM_SAMPLES]
        elif waveform.shape[1] < NUM_SAMPLES:
            waveform = torch.nn.functional.pad(waveform, (0, NUM_SAMPLES - waveform.shape[1]))

        waveform = torch.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0)
        waveform = waveform.clamp(-1.0, 1.0)
        return waveform

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        waveform = self._load_audio(row["audio_path"])
        label = torch.tensor(row["label"], dtype=torch.long)
        return waveform, label


train_dataset = FMASmallWaveformDataset(train_df, train=True)
val_dataset = FMASmallWaveformDataset(val_df)
test_dataset = FMASmallWaveformDataset(test_df)

# Keep the physical batch very small on low-memory GPUs; use gradient accumulation later to recover
# a larger effective batch size without allocating everything at once.
if LOW_VRAM_MODE:
    BATCH_SIZE = 2
    ACCUMULATION_STEPS = 16
elif device == "cuda":
    BATCH_SIZE = 8
    ACCUMULATION_STEPS = 4
else:
    BATCH_SIZE = 32
    ACCUMULATION_STEPS = 1
CPU_COUNT = os.cpu_count() or 1
NUM_WORKERS = min(8, max(2, CPU_COUNT // 2))
PIN_MEMORY = device == "cuda"

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(
    f"Device: {device} | Workers: {NUM_WORKERS} | "
    f"Batch size: {BATCH_SIZE} | Accumulation steps: {ACCUMULATION_STEPS} | "
    f"Low VRAM mode: {LOW_VRAM_MODE}"
)
batch_waveforms, batch_labels = next(iter(train_loader))
batch_waveforms.shape, batch_labels.shape

Device: cuda | Workers: 6 | Batch size: 2 | Accumulation steps: 16 | Low VRAM mode: True


[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!


(torch.Size([2, 1, 160000]), torch.Size([2]))

## GPU-Side Audio Preprocessing

In [45]:
class AudioPreprocessor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")

    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        specs = self.to_db(self.mel(waveforms))
        specs = torch.nan_to_num(specs, nan=0.0, posinf=0.0, neginf=0.0)
        mean = specs.mean(dim=(-2, -1), keepdim=True)
        std = specs.std(dim=(-2, -1), keepdim=True).clamp_min(1e-6)
        specs = (specs - mean) / std
        return torch.nan_to_num(specs, nan=0.0, posinf=0.0, neginf=0.0)


preprocessor = AudioPreprocessor().to(PREPROCESS_DEVICE)
example_waveform, example_label = train_dataset[0]
with torch.no_grad():
    example_spec = preprocessor(example_waveform.unsqueeze(0).to(PREPROCESS_DEVICE)).cpu().squeeze(0)
example_spec.shape, idx_to_genre[int(example_label)]

(torch.Size([1, 64, 1001]), 'Instrumental')

## Models

In [46]:
class SpectrogramCNN(nn.Module):
    def __init__(self, num_classes: int, channels: tuple[int, int, int, int] = (32, 64, 128, 256)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.features = nn.Sequential(
            nn.Conv2d(1, c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(c2, c3, kernel_size=3, padding=1),
            nn.BatchNorm2d(c3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(c3, c4, kernel_size=3, padding=1),
            nn.BatchNorm2d(c4),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(c4, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


class SpectrogramTransformer(nn.Module):
    def __init__(
        self,
        num_classes: int,
        input_shape: tuple[int, int, int],
        patch_size: tuple[int, int] = (16, 16),
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
    ):
        super().__init__()
        channels, n_mels, n_frames = input_shape
        self.patch_embed = nn.Conv2d(channels, d_model, kernel_size=patch_size, stride=patch_size)
        num_patches = (n_mels // patch_size[0]) * (n_frames // patch_size[1])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, d_model))
        self.pos_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.pos_dropout(x + self.pos_embed[:, :x.shape[1]])
        x = self.encoder(x)
        x = self.norm(x[:, 0])
        return self.head(x)


def build_cnn() -> nn.Module:
    channels = (16, 32, 64, 128) if LOW_VRAM_MODE else (32, 64, 128, 256)
    return SpectrogramCNN(num_classes=len(genres), channels=channels).to(device)


def build_transformer() -> nn.Module:
    transformer_kwargs = {
        "num_classes": len(genres),
        "input_shape": tuple(example_spec.shape),
    }
    if LOW_VRAM_MODE:
        transformer_kwargs.update({
            "d_model": 64,
            "nhead": 4,
            "num_layers": 2,
            "dim_feedforward": 128,
        })
    return SpectrogramTransformer(**transformer_kwargs).to(device)

## Training Utilities

In [47]:
# Mixed precision is faster on CUDA, but if you still see instability set this to False.
USE_AMP = device == "cuda"


def run_epoch(model, loader, criterion, optimizer=None, scaler=None, accumulation_steps: int = 1):
    is_train = optimizer is not None
    model.train(is_train)
    preprocessor.train(is_train)

    total_loss = 0.0
    correct = 0
    total = 0
    skipped_batches = 0
    optimizer_steps = 0

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    with torch.set_grad_enabled(is_train):
        for batch_index, (waveforms, labels) in enumerate(tqdm(loader, leave=False), start=1):
            labels = labels.to(device, non_blocking=True)
            waveforms = waveforms.to(PREPROCESS_DEVICE, non_blocking=False)
            specs = preprocessor(waveforms)
            specs = specs.to(device, non_blocking=True)
            if not torch.isfinite(specs).all():
                skipped_batches += 1
                continue

            amp_context = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
            with amp_context:
                logits = model(specs)
                loss = criterion(logits, labels)

            if not torch.isfinite(loss):
                skipped_batches += 1
                continue

            if is_train:
                scaled_loss = loss / accumulation_steps
                if USE_AMP:
                    scaler.scale(scaled_loss).backward()
                    should_step = (batch_index % accumulation_steps == 0) or (batch_index == len(loader))
                    if should_step:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad(set_to_none=True)
                        optimizer_steps += 1
                else:
                    scaled_loss.backward()
                    should_step = (batch_index % accumulation_steps == 0) or (batch_index == len(loader))
                    if should_step:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        optimizer.step()
                        optimizer.zero_grad(set_to_none=True)
                        optimizer_steps += 1

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        return float("nan"), float("nan"), skipped_batches, optimizer_steps
    return total_loss / total, correct / total, skipped_batches, optimizer_steps


def cleanup_cuda_memory(*objects):
    for obj in objects:
        del obj
    if device == "cuda":
        torch.cuda.empty_cache()


def train_model(name: str, build_model, epochs: int = 20, patience: int = 6, lr: float = 3e-4):
    model = build_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best_val_acc = 0.0
    epochs_without_improvement = 0
    best_model_path = PROJECT_DIR / f"best_{name}.pt"
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_skipped, optimizer_steps = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            accumulation_steps=ACCUMULATION_STEPS,
        )
        val_loss, val_acc, val_skipped, _ = run_epoch(model, val_loader, criterion)
        if optimizer_steps > 0:
            scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        history.append({
            "model": name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": current_lr,
            "train_skipped_batches": train_skipped,
            "val_skipped_batches": val_skipped,
        })

        if val_acc > best_val_acc + 1e-4:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            torch.save({"model_state_dict": model.state_dict()}, best_model_path)
        else:
            epochs_without_improvement += 1

        print(
            f"{name} | Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f}, acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f}, acc {val_acc:.3f} | lr {current_lr:.2e} | "
            f"skipped train/val batches {train_skipped}/{val_skipped}"
        )

        if epochs_without_improvement >= patience:
            print(f"{name} early stopping after {epoch} epochs.")
            break

    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    test_loss, test_acc, test_skipped, _ = run_epoch(model, test_loader, criterion)

    summary = {
        "model": name,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "epochs_ran": len(history),
        "test_skipped_batches": test_skipped,
    }
    return model, pd.DataFrame(history), summary

## Train the CNN Baseline

In [48]:
cleanup_cuda_memory()
cnn_model, cnn_history, cnn_summary = train_model("cnn_baseline", build_cnn, epochs=20, patience=6, lr=3e-4)
cnn_history.tail()

  0%|          | 0/1000 [00:00<?, ?it/s]

[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!


  0%|          | 0/400 [00:00<?, ?it/s]

cnn_baseline | Epoch 01 | train loss 2.0232, acc 0.204 | val loss 1.7969, acc 0.366 | lr 2.98e-04 | skipped train/val batches 0/0


  0%|          | 0/1000 [00:00<?, ?it/s]

...

[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!


  0%|          | 0/400 [00:00<?, ?it/s]

cnn_baseline | Epoch 20 | train loss 1.7878, acc 0.365 | val loss 1.6019, acc 0.446 | lr 0.00e+00 | skipped train/val batches 0/0


  0%|          | 0/400 [00:00<?, ?it/s]

,model,epoch,train_loss,train_acc,val_loss,val_acc,lr,train_skipped_batches,val_skipped_batches
15,cnn_baseline,16,1.801156,0.3525,1.630907,0.44625,0.000029,0,0
16,cnn_baseline,17,1.778422,0.3770,1.613383,0.46125,0.000016,0,0
17,cnn_baseline,18,1.780260,0.3500,1.641680,0.45375,0.000007,0,0
18,cnn_baseline,19,1.768036,0.3775,1.624751,0.43875,0.000002,0,0
19,cnn_baseline,20,1.787799,0.3650,1.601853,0.44625,0.000000,0,0


## Train the Transformer

In [49]:
cleanup_cuda_memory(cnn_model)
transformer_model, transformer_history, transformer_summary = train_model("transformer_baseline", build_transformer, epochs=20, patience=6, lr=3e-4)
transformer_history.tail()

/tmp/ipykernel_109906/3300194810.py:62: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


  0%|          | 0/1000 [00:00<?, ?it/s]

[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!


  0%|          | 0/400 [00:00<?, ?it/s]

transformer_baseline | Epoch 01 | train loss 1.9919, acc 0.226 | val loss 1.8205, acc 0.330 | lr 2.98e-04 | skipped train/val batches 0/0


...

[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!


  0%|          | 0/400 [00:00<?, ?it/s]

transformer_baseline | Epoch 20 | train loss 1.5029, acc 0.454 | val loss 1.6385, acc 0.386 | lr 0.00e+00 | skipped train/val batches 0/0


  0%|          | 0/400 [00:00<?, ?it/s]

,model,epoch,train_loss,train_acc,val_loss,val_acc,lr,train_skipped_batches,val_skipped_batches
15,transformer_baseline,16,1.521981,0.4390,1.641005,0.38500,0.000029,0,0
16,transformer_baseline,17,1.509318,0.4475,1.647554,0.39375,0.000016,0,0
17,transformer_baseline,18,1.518505,0.4515,1.638403,0.39250,0.000007,0,0
18,transformer_baseline,19,1.509821,0.4640,1.637497,0.38625,0.000002,0,0
19,transformer_baseline,20,1.502924,0.4540,1.638517,0.38625,0.000000,0,0


## Compare Results

In [ ]:
comparison = pd.DataFrame([cnn_summary, transformer_summary]).sort_values("best_val_acc", ascending=False)
comparison

In [ ]:
history = pd.concat([cnn_history, transformer_history], ignore_index=True)
history.pivot(index="epoch", columns="model", values="val_acc")